# Clasificación automática de residuos reciclables con transfer learning

**SI3014 – Redes Neuronales y Aprendizaje Profundo · Universidad EAFIT · Proyecto final**

**Objetivo general:** desarrollar un modelo de clasificación de imágenes basado en CNN preentrenadas (transfer learning) que identifique seis categorías de residuos sólidos (cartón, vidrio, metal, papel, plástico y residuo no reciclable) a partir de fotografías, alcanzando un F1-score macro de al menos 0,85 en el conjunto de prueba.

**Objetivos específicos**
1. Preparar TrashNet (2527 imágenes, 6 clases): redimensionamiento a 224 × 224, normalización ImageNet, *data augmentation* y división estratificada 70/15/15.
2. Entrenar y comparar dos estrategias de transfer learning sobre ResNet-18: extracción de características (backbone congelado) y *fine-tuning* de las últimas capas.
3. Evaluar el mejor modelo en el conjunto de prueba y en fotografías externas tomadas con celular, y analizar la variación del desempeño.

### Cómo usar este notebook
1. Abrirlo en **Google Colab** y activar GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. Ejecutar todo (*Entorno de ejecución → Ejecutar todas*). El dataset y los pesos de ResNet-18 se descargan solos.
3. Para la evaluación externa (sección 9), subir `externas.zip` con las fotos propias y volver a ejecutar esa sección.

Las figuras se guardan en la carpeta `figuras/` para usarlas en la presentación y el informe. El código base sigue el notebook de clase `Lectures/Lecture06/notebooks/TransferLearning/L06_CNN_Transfer.ipynb`.

## 0. Configuración

In [ ]:
# ===== Hiperparámetros y rutas =====
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

# Etapa A: extracción de características (solo se entrena la cabeza)
EPOCHS_FE = 10
LR_FE = 1e-3

# Etapa B: fine-tuning (se descongela layer4 de ResNet-18)
EPOCHS_FT = 15
LR_FT_BACKBONE = 1e-4
LR_FT_HEAD = 5e-4

WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
USAR_PESOS_CLASE = True   # compensa el desbalance (la clase 'trash' tiene ~137 imágenes)

DATA_DIR = 'data'
EXT_DIR = 'externas'       # fotos propias: externas/<clase>/*.jpg
FIG_DIR = 'figuras'

MODO_RAPIDO = False        # True = 30 imágenes por clase y 1 época por etapa, solo para probar que todo corre

In [ ]:
import os, glob, time, copy, random, zipfile, platform, urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

if MODO_RAPIDO:
    EPOCHS_FE, EPOCHS_FT = 1, 1

# En Windows los DataLoader con workers fallan dentro de notebooks; en Colab (Linux) sí se usan
NUM_WORKERS = 2 if platform.system() == 'Linux' else 0

def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fijar_semilla(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(FIG_DIR, exist_ok=True)
print('Dispositivo:', device)

### Recursos computacionales
Esta información se pide explícitamente en la sustentación (CPU, GPU, RAM y entorno de ejecución).

In [ ]:
def info_hardware():
    info = {
        'Entorno': 'Google Colab' if 'COLAB_RELEASE_TAG' in os.environ else platform.platform(),
        'Python': platform.python_version(),
        'PyTorch': torch.__version__,
        'CPU (núcleos lógicos)': os.cpu_count(),
    }
    try:
        import psutil
        info['RAM (GB)'] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        pass
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info['GPU'] = props.name
        info['VRAM (GB)'] = round(props.total_memory / 1e9, 1)
    else:
        info['GPU'] = 'No disponible (CPU)'
    return info

HARDWARE = info_hardware()
for k, v in HARDWARE.items():
    print(f'{k}: {v}')

## 1. Base de datos: TrashNet

[TrashNet](https://github.com/garythung/trashnet) (Thung y Yang, 2016) contiene 2527 fotografías de residuos sobre fondo blanco, redimensionadas a 512 × 384 píxeles, en seis clases: `cardboard`, `glass`, `metal`, `paper`, `plastic` y `trash` (residuo no reciclable).

> Referencia: G. Thung y M. Yang, *Classification of Trash for Recyclability Status*, CS229 Project Report, Stanford University, 2016.

In [ ]:
URL = 'https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip'
ROOT = os.path.join(DATA_DIR, 'dataset-resized')
zip_path = os.path.join(DATA_DIR, 'dataset-resized.zip')

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(zip_path):
    print('Descargando TrashNet (~43 MB)...')
    urllib.request.urlretrieve(URL, zip_path)
if not os.path.isdir(ROOT):
    with zipfile.ZipFile(zip_path) as z:
        # Se omiten los metadatos de macOS que trae el zip
        miembros = [m for m in z.namelist() if not m.startswith('__MACOSX')]
        z.extractall(DATA_DIR, members=miembros)
print('Dataset en', ROOT)

Igual que en el notebook de clase, se verifica que no haya imágenes corruptas antes de construir el `Dataset`:

In [ ]:
def check_image_dir(pattern):
    # Elimina las imágenes corruptas que coincidan con el patrón
    eliminadas = 0
    for fname in glob.glob(pattern):
        try:
            with Image.open(fname) as img:
                img.verify()
        except Exception:
            os.remove(fname)
            eliminadas += 1
    return eliminadas

CLASES = sorted(d for d in os.listdir(ROOT)
                if os.path.isdir(os.path.join(ROOT, d)) and not d.startswith(('.', '_')))
NOMBRES_ES = {'cardboard': 'cartón', 'glass': 'vidrio', 'metal': 'metal',
              'paper': 'papel', 'plastic': 'plástico', 'trash': 'no reciclable'}
ETIQUETAS_ES = [NOMBRES_ES.get(c, c) for c in CLASES]

for c in CLASES:
    n = check_image_dir(os.path.join(ROOT, c, '*.jpg'))
    if n:
        print(f'{c}: {n} imágenes corruptas eliminadas')

rutas, etiquetas = [], []
for i, c in enumerate(CLASES):
    archivos = sorted(glob.glob(os.path.join(ROOT, c, '*.jpg')))
    if MODO_RAPIDO:
        archivos = archivos[:30]
    rutas += archivos
    etiquetas += [i] * len(archivos)

print('Clases:', CLASES)
print('Total de imágenes:', len(rutas))

## 2. Análisis exploratorio

In [ ]:
conteo = pd.Series(etiquetas).value_counts().sort_index()
conteo.index = ETIQUETAS_ES

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(conteo.index, conteo.values, color='#4C72B0')
ax.bar_label(barras)
ax.set_title('Número de imágenes por clase (TrashNet)')
ax.set_ylabel('Imágenes')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/distribucion_clases.png', dpi=150)
plt.show()

print(f'Desbalance: la clase mayoritaria tiene {conteo.max() / conteo.min():.1f} veces más imágenes que la minoritaria.')
with Image.open(rutas[0]) as img:
    print('Tamaño de las imágenes originales:', img.size)

In [ ]:
fig, axes = plt.subplots(len(CLASES), 5, figsize=(12, 2.2 * len(CLASES)))
rng = np.random.default_rng(SEED)
for i, c in enumerate(CLASES):
    idx = [k for k, e in enumerate(etiquetas) if e == i]
    for j, k in enumerate(rng.choice(idx, size=min(5, len(idx)), replace=False)):
        axes[i, j].imshow(Image.open(rutas[k]))
        axes[i, j].axis('off')
    axes[i, 0].set_title(ETIQUETAS_ES[i], loc='left', fontsize=11)
plt.suptitle('Ejemplos por clase', y=1.0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/ejemplos_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. División en entrenamiento, validación y prueba

División **estratificada** 70/15/15: cada conjunto conserva la proporción de clases, lo que es importante porque `trash` es minoritaria. La semilla fija hace que la división sea reproducible.

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    rutas, etiquetas, test_size=0.30, stratify=etiquetas, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

tabla_split = pd.DataFrame({
    'entrenamiento': np.bincount(y_train, minlength=len(CLASES)),
    'validación': np.bincount(y_val, minlength=len(CLASES)),
    'prueba': np.bincount(y_test, minlength=len(CLASES)),
}, index=ETIQUETAS_ES)
tabla_split.loc['total'] = tabla_split.sum()
tabla_split

## 4. Preprocesamiento y aumento de datos

- **Normalización** con la media y desviación estándar de ImageNet, porque ResNet-18 fue preentrenada con esas estadísticas.
- **Entrenamiento:** recorte aleatorio con reescalado (`RandomResizedCrop`), volteos horizontales y verticales y rotaciones (un residuo no tiene una orientación canónica), y variaciones de brillo, contraste y saturación para simular distintas condiciones de iluminación.
- **Validación, prueba y datos externos:** solo redimensionamiento a 256, recorte central de 224 y normalización, sin aleatoriedad.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class ResiduosDataset(Dataset):
    def __init__(self, rutas, etiquetas, transform):
        self.rutas, self.etiquetas, self.transform = rutas, etiquetas, transform

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        # exif_transpose corrige la orientación de las fotos tomadas con celular
        img = ImageOps.exif_transpose(Image.open(self.rutas[i])).convert('RGB')
        return self.transform(img), self.etiquetas[i]

def desnormalizar(x):
    # Tensor normalizado [3, H, W] -> imagen [H, W, 3] en [0, 1] para graficar
    m, s = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
    return (x.cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

train_ds = ResiduosDataset(X_train, y_train, train_tf)
val_ds = ResiduosDataset(X_val, y_val, eval_tf)
test_ds = ResiduosDataset(X_test, y_test, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

In [ ]:
# Visualización del aumento de datos sobre una misma imagen
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
original = Image.open(X_train[0]).convert('RGB')
axes[0].imshow(original)
axes[0].set_title('Original')
for ax in axes[1:]:
    ax.imshow(desnormalizar(train_tf(original)))
    ax.set_title('Aumentada')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Modelo: ResNet-18 preentrenada en ImageNet

> Referencia: K. He, X. Zhang, S. Ren y J. Sun, *Deep Residual Learning for Image Recognition*, CVPR 2016.

ResNet-18 tiene 18 capas con conexiones residuales, que facilitan el flujo del gradiente (ver `l06-gradient-flow-residual-connections` del curso). Es liviana (~11,7 M de parámetros), por lo que entrena rápido en una GPU gratuita, y sus filtros aprendidos en ImageNet (bordes, texturas, formas) son útiles para distinguir materiales.

**Modificación respecto al modelo original:** se reemplaza la capa final `fc` (1000 clases de ImageNet) por `Dropout(0.3) + Linear(512, 6)`.

**Estrategia en dos etapas**, igual que en el notebook de clase:
- **Etapa A – extracción de características:** todo el backbone congelado; solo se entrena la nueva cabeza.
- **Etapa B – fine-tuning:** partiendo del modelo de la etapa A, se descongela el último bloque (`layer4`) y se entrena con una tasa de aprendizaje menor para no destruir los pesos preentrenados.

In [ ]:
def crear_modelo():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(model.fc.in_features, len(CLASES)))
    return model.to(device)

def contar_parametros(model):
    total = sum(p.numel() for p in model.parameters())
    entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, entrenables

if USAR_PESOS_CLASE:
    # Peso inversamente proporcional a la frecuencia de cada clase en entrenamiento
    frec = np.bincount(y_train, minlength=len(CLASES))
    pesos = torch.tensor(len(y_train) / (len(CLASES) * frec), dtype=torch.float32)
    print('Pesos por clase:', {e: round(float(p), 2) for e, p in zip(ETIQUETAS_ES, pesos)})
    criterion = nn.CrossEntropyLoss(weight=pesos.to(device))
else:
    criterion = nn.CrossEntropyLoss()

## 6. Funciones de entrenamiento y evaluación

- **Pérdida:** entropía cruzada (con pesos por clase si `USAR_PESOS_CLASE`).
- **Optimizador:** AdamW con *weight decay* y *scheduler* coseno.
- **Selección del modelo:** se guardan los pesos de la época con mejor **F1 macro en validación** (nunca se usa el conjunto de prueba para decidir).
- Las métricas de entrenamiento se calculan con aumento de datos y dropout activos, por eso pueden quedar por debajo de las de validación en las primeras épocas.

In [ ]:
def evaluar(model, loader, criterion=None):
    model.eval()
    perdida, y_true, probs = 0.0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            if criterion is not None:
                perdida += criterion(out, y).item() * x.size(0)
            probs.append(out.softmax(1).cpu())
            y_true += y.tolist()
    probs = torch.cat(probs).numpy()
    y_true, y_pred = np.array(y_true), probs.argmax(1)
    return {'loss': perdida / len(y_true) if criterion is not None else None,
            'acc': accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, average='macro'),
            'y_true': y_true, 'y_pred': y_pred, 'probs': probs}

def entrenar(model, optimizer, epochs, scheduler=None, nombre='modelo'):
    hist = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc', 'train_f1', 'val_f1', 'lr']}
    mejor_f1, mejor_ep, mejores_pesos = -1.0, 0, None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        # Las BatchNorm congeladas se dejan en modo eval para no alterar sus estadísticas de ImageNet
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        perdida, y_true, y_pred = 0.0, [], []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            perdida += loss.item() * x.size(0)
            y_true += y.tolist()
            y_pred += out.argmax(1).tolist()
        hist['lr'].append(optimizer.param_groups[-1]['lr'])
        if scheduler is not None:
            scheduler.step()

        val = evaluar(model, val_loader, criterion)
        hist['train_loss'].append(perdida / len(y_true))
        hist['train_acc'].append(accuracy_score(y_true, y_pred))
        hist['train_f1'].append(f1_score(y_true, y_pred, average='macro'))
        hist['val_loss'].append(val['loss'])
        hist['val_acc'].append(val['acc'])
        hist['val_f1'].append(val['f1'])
        if val['f1'] > mejor_f1:
            mejor_f1, mejor_ep = val['f1'], ep
            mejores_pesos = copy.deepcopy(model.state_dict())
        print(f'[{nombre}] Época {ep:02d}/{epochs} | '
              f'train loss {hist["train_loss"][-1]:.4f} acc {hist["train_acc"][-1]:.3f} | '
              f'val loss {val["loss"]:.4f} acc {val["acc"]:.3f} F1 {val["f1"]:.3f}')

    duracion = time.time() - t0
    model.load_state_dict(mejores_pesos)
    print(f'Tiempo de entrenamiento: {duracion / 60:.1f} min | mejor F1 val = {mejor_f1:.3f} (época {mejor_ep})')
    return hist, {'tiempo_min': duracion / 60, 'mejor_f1_val': mejor_f1, 'mejor_epoca': mejor_ep}

def graficar_curvas(hist, titulo, archivo):
    epocas = range(1, len(hist['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, k, nombre in zip(axes, ['loss', 'acc', 'f1'], ['Pérdida', 'Accuracy', 'F1 macro']):
        ax.plot(epocas, hist[f'train_{k}'], 'o-', label='entrenamiento')
        ax.plot(epocas, hist[f'val_{k}'], 'o-', label='validación')
        ax.set_title(nombre)
        ax.set_xlabel('Época')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{archivo}', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Entrenamiento

### Etapa A: extracción de características (backbone congelado)

In [ ]:
fijar_semilla(SEED)
model_fe = crear_modelo()
total, entrenables = contar_parametros(model_fe)
print(f'Parámetros totales: {total:,} | entrenables: {entrenables:,}')

opt_fe = torch.optim.AdamW(model_fe.fc.parameters(), lr=LR_FE, weight_decay=WEIGHT_DECAY)
sched_fe = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fe, T_max=EPOCHS_FE)
hist_fe, res_fe = entrenar(model_fe, opt_fe, EPOCHS_FE, sched_fe, nombre='A: feature extraction')
res_fe['parametros_entrenables'] = entrenables

In [ ]:
graficar_curvas(hist_fe, 'Etapa A – extracción de características', 'curvas_etapaA.png')

**Análisis (completar con los resultados obtenidos):** ¿la pérdida de validación baja junto con la de entrenamiento? ¿Hay señales de sobreajuste (la pérdida de validación sube mientras la de entrenamiento baja) o de subajuste (ambas se estancan en valores altos)?

### Etapa B: fine-tuning de `layer4`

In [ ]:
fijar_semilla(SEED)
model_ft = copy.deepcopy(model_fe)
for p in model_ft.layer4.parameters():
    p.requires_grad = True
total, entrenables = contar_parametros(model_ft)
print(f'Parámetros totales: {total:,} | entrenables: {entrenables:,}')

# Tasas de aprendizaje diferenciadas: menor para el backbone preentrenado, mayor para la cabeza
opt_ft = torch.optim.AdamW([
    {'params': model_ft.layer4.parameters(), 'lr': LR_FT_BACKBONE},
    {'params': model_ft.fc.parameters(), 'lr': LR_FT_HEAD},
], weight_decay=WEIGHT_DECAY)
sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=EPOCHS_FT)
hist_ft, res_ft = entrenar(model_ft, opt_ft, EPOCHS_FT, sched_ft, nombre='B: fine-tuning')
res_ft['parametros_entrenables'] = entrenables

In [ ]:
graficar_curvas(hist_ft, 'Etapa B – fine-tuning de layer4', 'curvas_etapaB.png')

### Comparación de estrategias (sobre validación)

In [ ]:
comparacion = pd.DataFrame([
    {'estrategia': 'A: extracción de características', 'épocas': EPOCHS_FE, **res_fe},
    {'estrategia': 'B: fine-tuning layer4', 'épocas': EPOCHS_FT, **res_ft},
]).set_index('estrategia').round(3)
display(comparacion)

# El modelo final se elige con validación, no con prueba
if res_ft['mejor_f1_val'] >= res_fe['mejor_f1_val']:
    mejor_modelo, nombre_mejor = model_ft, 'B: fine-tuning layer4'
else:
    mejor_modelo, nombre_mejor = model_fe, 'A: extracción de características'
torch.save(mejor_modelo.state_dict(), 'modelo_residuos.pth')
print('Modelo seleccionado:', nombre_mejor, '-> guardado en modelo_residuos.pth')

## 8. Resultados en el conjunto de prueba

In [ ]:
def reporte(res, titulo, archivo):
    labels = list(range(len(CLASES)))
    print(f'{titulo}\nAccuracy: {res["acc"]:.3f} | F1 macro: {res["f1"]:.3f}\n')
    print(classification_report(res['y_true'], res['y_pred'], labels=labels,
                                target_names=ETIQUETAS_ES, digits=3, zero_division=0))
    cm = confusion_matrix(res['y_true'], res['y_pred'], labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ETIQUETAS_ES, yticklabels=ETIQUETAS_ES)
    plt.xlabel('Predicción')
    plt.ylabel('Clase real')
    plt.title(f'Matriz de confusión – {titulo}')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{archivo}', dpi=150, bbox_inches='tight')
    plt.show()

res_test = evaluar(mejor_modelo, test_loader)
reporte(res_test, 'Prueba (TrashNet)', 'matriz_confusion_prueba.png')

In [ ]:
def mostrar_predicciones(dataset, res, titulo, archivo, solo_errores=False, n=12):
    idx = np.arange(len(res['y_true']))
    if solo_errores:
        idx = idx[res['y_true'] != res['y_pred']]
    if len(idx) == 0:
        print('No hay ejemplos para mostrar.')
        return
    idx = idx[:n]
    cols = 6
    filas = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(filas, cols, figsize=(2.6 * cols, 3 * filas), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, i in zip(axes.flat, idx):
        x, _ = dataset[i]
        real, pred = res['y_true'][i], res['y_pred'][i]
        ax.imshow(desnormalizar(x))
        ax.set_title(f'real: {ETIQUETAS_ES[real]}\npred: {ETIQUETAS_ES[pred]} ({res["probs"][i, pred]:.0%})',
                     fontsize=9, color='green' if real == pred else 'red')
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{archivo}', dpi=150, bbox_inches='tight')
    plt.show()

mostrar_predicciones(test_ds, res_test, 'Errores en el conjunto de prueba', 'errores_prueba.png', solo_errores=True)

## 9. Evaluación con datos externos (generalización)

TrashNet tiene fotos sobre **fondo blanco y con iluminación controlada**. Para medir la generalización se usan fotos propias en condiciones reales.

**Instrucciones**
1. Tomar con el celular entre 15 y 25 fotos por clase de residuos de la casa o la universidad, con fondos e iluminación variados.
2. Organizarlas en carpetas con los **mismos nombres en inglés** del dataset:
   ```
   externas/
     cardboard/  glass/  metal/  paper/  plastic/  trash/
   ```
3. Comprimir la carpeta como `externas.zip`, subirla a Colab (panel *Archivos*) y ejecutar esta sección.

In [ ]:
EXTENSIONES = ('.jpg', '.jpeg', '.png', '.webp')

if os.path.exists('externas.zip') and not os.path.isdir(EXT_DIR):
    with zipfile.ZipFile('externas.zip') as z:
        z.extractall('.')

ext_rutas, ext_etiquetas = [], []
for i, c in enumerate(CLASES):
    carpeta = os.path.join(EXT_DIR, c)
    if os.path.isdir(carpeta):
        for f in sorted(os.listdir(carpeta)):
            if f.lower().endswith(EXTENSIONES):
                ext_rutas.append(os.path.join(carpeta, f))
                ext_etiquetas.append(i)

HAY_EXTERNAS = len(ext_rutas) > 0
if HAY_EXTERNAS:
    print('Fotos externas por clase:', dict(zip(ETIQUETAS_ES, np.bincount(ext_etiquetas, minlength=len(CLASES)).tolist())))
else:
    print(f'No se encontraron fotos en {EXT_DIR}/. Sube externas.zip y vuelve a ejecutar esta sección.')

In [ ]:
if HAY_EXTERNAS:
    ext_ds = ResiduosDataset(ext_rutas, ext_etiquetas, eval_tf)
    ext_loader = DataLoader(ext_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    res_ext = evaluar(mejor_modelo, ext_loader)
    reporte(res_ext, 'Fotos externas', 'matriz_confusion_externas.png')

    generalizacion = pd.DataFrame([
        {'conjunto': 'Prueba (TrashNet)', 'imágenes': len(res_test['y_true']), 'accuracy': res_test['acc'], 'F1 macro': res_test['f1']},
        {'conjunto': 'Externas (celular)', 'imágenes': len(res_ext['y_true']), 'accuracy': res_ext['acc'], 'F1 macro': res_ext['f1']},
    ]).set_index('conjunto').round(3)
    display(generalizacion)

    mostrar_predicciones(ext_ds, res_ext, 'Predicciones sobre fotos externas', 'predicciones_externas.png', n=18)

**Análisis (completar):** ¿se mantiene el rendimiento o se degrada? Causas posibles: fondo distinto al blanco de TrashNet, varios objetos en la misma foto, iluminación, objetos sucios o deformados, clases ambiguas (papel frente a cartón, plástico frente a vidrio transparente). La sección 10 ayuda a comprobarlo.

## 10. Interpretabilidad con Grad-CAM

Grad-CAM (Selvaraju et al., 2017) muestra qué regiones de la imagen usó el modelo para decidir. Sirve para comprobar si el modelo mira el objeto o el fondo, sobre todo en los errores con fotos externas.

> Referencia: R. R. Selvaraju et al., *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*, ICCV 2017.

In [ ]:
def grad_cam(model, x):
    # x: tensor [3, H, W] normalizado. Devuelve el mapa de calor [H, W] en [0, 1] y la clase predicha
    model.eval()
    guardado = {}

    def hook(modulo, entrada, salida):
        guardado['act'] = salida
        salida.register_hook(lambda g: guardado.update(grad=g))

    h = model.layer4.register_forward_hook(hook)
    x = x.unsqueeze(0).to(device).requires_grad_(True)
    out = model(x)
    clase = out.argmax(1).item()
    model.zero_grad()
    out[0, clase].backward()
    h.remove()

    pesos = guardado['grad'].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((pesos * guardado['act']).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy(), clase

def mostrar_grad_cam(dataset, indices, titulo, archivo):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.8 * len(indices), 6), squeeze=False)
    for j, i in enumerate(indices):
        x, real = dataset[i]
        cam, pred = grad_cam(mejor_modelo, x)
        img = desnormalizar(x)
        axes[0, j].imshow(img)
        axes[0, j].set_title(f'real: {ETIQUETAS_ES[real]}', fontsize=9)
        axes[1, j].imshow(img)
        axes[1, j].imshow(cam, cmap='jet', alpha=0.45)
        axes[1, j].set_title(f'pred: {ETIQUETAS_ES[pred]}', fontsize=9,
                             color='green' if real == pred else 'red')
    for ax in axes.flat:
        ax.axis('off')
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{archivo}', dpi=150, bbox_inches='tight')
    plt.show()

errores = np.where(res_test['y_true'] != res_test['y_pred'])[0]
aciertos = np.where(res_test['y_true'] == res_test['y_pred'])[0]
mostrar_grad_cam(test_ds, list(aciertos[:3]) + list(errores[:3]), 'Grad-CAM en prueba (aciertos y errores)', 'gradcam_prueba.png')

if HAY_EXTERNAS:
    mostrar_grad_cam(ext_ds, list(range(min(6, len(ext_ds)))), 'Grad-CAM en fotos externas', 'gradcam_externas.png')

## 11. Predicción de una imagen nueva (demo para la sustentación)

Sube cualquier foto a Colab y cambia la ruta.

In [ ]:
def predecir(ruta, top=3):
    img = ImageOps.exif_transpose(Image.open(ruta)).convert('RGB')
    x = eval_tf(img).unsqueeze(0).to(device)
    mejor_modelo.eval()
    with torch.no_grad():
        p = mejor_modelo(x).softmax(1)[0].cpu().numpy()
    orden = p.argsort()[::-1][:top]
    plt.imshow(img)
    plt.axis('off')
    plt.title(' | '.join(f'{ETIQUETAS_ES[k]} {p[k]:.0%}' for k in orden))
    plt.show()

ruta_demo = ext_rutas[0] if HAY_EXTERNAS else X_test[0]   # cambiar por la ruta de cualquier foto
predecir(ruta_demo)

## 12. Resumen para el informe y la presentación

In [ ]:
resumen = {
    **HARDWARE,
    'Modelo': 'ResNet-18 (ImageNet) + Dropout + Linear(512, 6)',
    'Modelo seleccionado': nombre_mejor,
    'Imágenes train / val / test': f'{len(X_train)} / {len(X_val)} / {len(X_test)}',
    'Batch size': BATCH_SIZE,
    'Optimizador': f'AdamW (weight decay {WEIGHT_DECAY}) + CosineAnnealingLR',
    'Etapa A': f'{EPOCHS_FE} épocas, lr {LR_FE}, {res_fe["tiempo_min"]:.1f} min',
    'Etapa B': f'{EPOCHS_FT} épocas, lr backbone {LR_FT_BACKBONE} / cabeza {LR_FT_HEAD}, {res_ft["tiempo_min"]:.1f} min',
    'Pesos por clase': USAR_PESOS_CLASE,
    'Accuracy prueba': round(res_test['acc'], 3),
    'F1 macro prueba': round(res_test['f1'], 3),
    'Meta F1 macro >= 0,85': 'cumplida' if res_test['f1'] >= 0.85 else 'no cumplida',
}
if HAY_EXTERNAS:
    resumen['Accuracy externas'] = round(res_ext['acc'], 3)
    resumen['F1 macro externas'] = round(res_ext['f1'], 3)

pd.set_option('display.max_colwidth', None)
pd.Series(resumen, name='valor').to_frame()

In [ ]:
# Empaqueta las figuras y el modelo para descargarlos (en Colab se descargan automáticamente)
import shutil
shutil.copy('modelo_residuos.pth', FIG_DIR)
shutil.make_archive('resultados', 'zip', FIG_DIR)
try:
    from google.colab import files
    files.download('resultados.zip')
except ImportError:
    print('resultados.zip creado en la carpeta del notebook')